In [2]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/raw/Solar_Weather_Data_Processed.csv")
df.columns = df.columns.str.strip()
df["timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

WEATHER = ["GHI", "Temperature", "Relative Humidity", "Wind Speed", "Wind Direction"]
TARGET = "AC Output"

# Clock correction chosen in notebook 03 (-10 h); the shift is fixed, not tuned.
df[WEATHER] = df[WEATHER].shift(-10)
df = df.dropna(subset=WEATHER).reset_index(drop=True)

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
print("Train:", len(train), "| Test:", len(test))

Train: 2213 | Test: 554


In [3]:
def cv_daylight(cols):
    """Mean and spread of daylight MAE across 5 time-ordered folds of the training data."""
    rf_maes, base_maes = [], []
    for tr_i, va_i in TimeSeriesSplit(n_splits=5).split(train):
        tr_, va_ = train.iloc[tr_i], train.iloc[va_i]
        va_day = va_[va_[TARGET] > 0]

        m = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        m.fit(tr_[cols], tr_[TARGET])
        rf_maes.append(mean_absolute_error(va_day[TARGET], m.predict(va_day[cols])))

        hm = tr_.groupby("Hour")[TARGET].mean()
        base_maes.append(mean_absolute_error(va_day[TARGET], va_day["Hour"].map(hm)))
    return np.mean(rf_maes), np.std(rf_maes), np.mean(base_maes)

rows = []
for name, cols in {
    "Hour": ["Hour"],
    "Hour + Month": ["Hour", "Month"],
    "Hour + Month + solar angles": ["Hour", "Month", "SZ Angle", "SA Angle"],
    "Hour + Month + solar angles + GHI": ["Hour", "Month", "SZ Angle", "SA Angle", "GHI"],
}.items():
    rf_mean, rf_sd, base = cv_daylight(cols)
    rows.append({"Features": name, "RF MAE": round(rf_mean, 1),
                 "RF fold SD": round(rf_sd, 1), "Baseline MAE": round(base, 1)})

print(pd.DataFrame(rows).to_string(index=False))

                         Features  RF MAE  RF fold SD  Baseline MAE
                             Hour   108.9         4.8         108.9
                     Hour + Month   115.6        13.9         108.9
      Hour + Month + solar angles   128.8        14.4         108.9
Hour + Month + solar angles + GHI   131.6        12.9         108.9


In [4]:
CHOSEN = ["Hour"]
grid = [(d, l) for d in [None, 8, 12] for l in [1, 5, 20]]

def cv_mae(depth, leaf):
    maes = []
    for tr_i, va_i in TimeSeriesSplit(n_splits=5).split(train):
        tr_, va_ = train.iloc[tr_i], train.iloc[va_i]
        va_day = va_[va_[TARGET] > 0]
        m = RandomForestRegressor(n_estimators=200, max_depth=depth,
                                  min_samples_leaf=leaf, random_state=42, n_jobs=-1)
        m.fit(tr_[CHOSEN], tr_[TARGET])
        maes.append(mean_absolute_error(va_day[TARGET], m.predict(va_day[CHOSEN])))
    return np.mean(maes)

best_depth, best_leaf = min(grid, key=lambda p: cv_mae(*p))
print("Best by CV: max_depth =", best_depth, "| min_samples_leaf =", best_leaf)

# Test set, used once
best = RandomForestRegressor(n_estimators=200, max_depth=best_depth,
                             min_samples_leaf=best_leaf, random_state=42, n_jobs=-1)
best.fit(train[CHOSEN], train[TARGET])

day = test[test[TARGET] > 0]
hm = train.groupby("Hour")[TARGET].mean()

def row(name, p):
    return {"Model": name,
            "MAE": round(mean_absolute_error(day[TARGET], p), 1),
            "RMSE": round(np.sqrt(mean_squared_error(day[TARGET], p)), 1),
            "R2": round(r2_score(day[TARGET], p), 3)}

final = pd.DataFrame([row("Hourly average (baseline)", day["Hour"].map(hm)),
                      row("Random Forest, Hour only (tuned)", best.predict(day[CHOSEN]))])
print(final.to_string(index=False))

Best by CV: max_depth = 12 | min_samples_leaf = 5
                           Model   MAE  RMSE    R2
       Hourly average (baseline) 107.1 149.4 0.832
Random Forest, Hour only (tuned) 107.1 149.5 0.832


In [5]:
import joblib
hourly_lookup = train.groupby("Hour")[TARGET].mean()
joblib.dump(hourly_lookup, "../models/hourly_lookup.joblib")
#final.to_csv("../models/metrics.csv", index=False)
print(hourly_lookup.round(1).to_string())

Hour
0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
5       0.0
6      58.7
7     244.9
8     502.4
9     726.0
10    840.6
11    991.1
12    958.8
13    865.8
14    760.9
15    666.4
16    414.1
17    170.5
18     22.2
19      0.0
20      0.0
21      0.0
22      0.0
23      0.0
